# QQQ Systematic Signal Research

This notebook is the polished walkthrough for the committed results in this repository. It is intentionally light on exploratory clutter and focuses on the exact evidence a reader can verify quickly:

1. data split
2. no-lookahead design
3. signal selection
4. final performance table
5. cost and parameter sensitivity
6. regime analysis
7. limitations


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
  ROOT = ROOT.parent

results_dir = ROOT / "results"
figures_dir = ROOT / "figures"

split_df = pd.read_csv(results_dir / "public_data_split.csv")
final_df = pd.read_csv(results_dir / "final_performance_summary.csv")
selection_df = pd.read_csv(results_dir / "model_selection_summary.csv")
cost_df = pd.read_csv(results_dir / "cost_sensitivity.csv")
param_df = pd.read_csv(results_dir / "parameter_sensitivity.csv")
regime_df = pd.read_csv(results_dir / "regime_summary.csv")


## Data And Repo Split

The committed included dataset spans 2018-01-02 through 2025-06-30. Because the original full-history challenge dataset is not included in the repo, the committed dataset uses a shorter but fully reproducible train / validation / holdout split.

In [ ]:
display(split_df)

## Backtest Integrity

The core backtest logic applies exposure to the **next** close-to-close return rather than the current one. In the codebase this is implemented by shifting the exposure series before returns are applied. The holdout period is not used for iterative signal selection in the committed results.

## Final Performance Table

This table is the shortest summary of the committed results.

In [ ]:
display(final_df)

## Model Selection

Selection is validation-driven and family-aware. The committed repo ensemble keeps the strongest family representatives under a validation-only score while removing near-duplicate validation profiles.

In [ ]:
display(selection_df)

## Cost Sensitivity

The final ensemble is not immune to trading friction. The committed cost-sensitivity table makes that explicit rather than burying it.

In [ ]:
display(cost_df[(cost_df["strategy"] == "final_research_ensemble") & (cost_df["split"] == "holdout")][["cost_bps", "sharpe", "ann_return", "max_drawdown"]])

## Parameter Sensitivity

The included dataset shows the clearest sensitivity in the trend sleeve. Some other sleeves are effectively invariant on this shorter repo dataset, which is itself an important limitation.

In [ ]:
display(param_df[param_df["split"] == "holdout"].groupby("signal")["sharpe"].agg(["min", "max", "mean"]).reset_index())

## Regime Breakdown

The structural regime summary is useful for checking whether the final ensemble is simply a disguised bull-market lever or whether it actually changes drawdown behavior.

In [ ]:
structural = regime_df[(regime_df["regime_set"] == "structural") & (regime_df["strategy"].isin(["buy_and_hold", "final_research_ensemble"]))]
display(structural[["strategy", "regime", "ann_return", "sharpe", "max_drawdown"]])

## Figures

The final static figures are committed directly to the repo so readers do not have to run anything to inspect the project.

In [ ]:
display(Image(filename=str(figures_dir / "final_equity_curve.png")))
display(Image(filename=str(figures_dir / "final_drawdown.png")))
display(Image(filename=str(figures_dir / "rolling_sharpe.png")))
display(Image(filename=str(figures_dir / "cost_sensitivity.png")))
display(Image(filename=str(figures_dir / "parameter_sensitivity_heatmap.png")))
display(Image(filename=str(figures_dir / "regime_breakdown.png")))

## Limitations

- The included dataset is shorter than the intended full-history research design.
- The current repo results are useful for verifying process quality, but they are not a substitute for the full original research stack.
- Holdout performance on this included sample is mixed, which is exactly why the repo includes explicit cost, sensitivity, and regime summaries instead of only a headline Sharpe.
